In [1]:
from qiskit_metal import designs, Dict
from qiskit_metal.qlibrary.tlines.meandered_grounded import RouteMeanderGrounded
from qiskit_metal.qlibrary.tlines.meandered import RouteMeander
from qiskit_metal.qlibrary.terminations.open_to_ground import OpenToGround
from qiskit_metal.qlibrary.terminations.short_to_ground import ShortToGround
from qiskit_metal.qlibrary.terminations.launchpad_wb import LaunchpadWirebond
from qiskit_metal.qlibrary.tlines.straight_path import RouteStraight
from qiskit_metal import MetalGUI

In [9]:
x_min = -0.6144
x_max = 0.66144
y_min = -0.91
y_max = 0.0123

In [10]:
# resonator parameters
TOTAL_LENGTH_MM = 5.92905
GROUND_WIDTH_UM = 10
GROUND_OVERLAP_UM = 4
EDGE_MARGIN_MM = 0.5

# chip geometry (make it as tight as possible)
CHIP_SIZE_X_MM = x_max - x_min
CHIP_SIZE_Y_MM = y_max - y_min
CHIP_SIZE_Z_UM = 500
CHIP_CENTER_X_MM = (x_max + x_min)/2
CHIP_CENTER_Y_MM = (y_max + y_min)/2

In [3]:
ground_pos_mm = 2.0 * TOTAL_LENGTH_MM / 3.0
total_length_mm = TOTAL_LENGTH_MM

In [4]:
chip_size_x_mm = CHIP_SIZE_X_MM
chip_size_y_mm = CHIP_SIZE_Y_MM
chip_center_x_mm = CHIP_CENTER_X_MM
chip_center_y_mm = CHIP_CENTER_Y_MM
ground_overlap_um = GROUND_OVERLAP_UM

design = designs.DesignPlanar({}, overwrite_enabled=True)

design.chips.main.size.size_x = f'{chip_size_x_mm}mm'
design.chips.main.size.size_y = f'{chip_size_y_mm}mm'
design.chips.main.size.size_z = f'{CHIP_SIZE_Z_UM}um'
design.chips.main.size.center_x = f'{chip_center_x_mm}mm'
design.chips.main.size.center_y = f'{chip_center_y_mm}mm'

design.variables['cpw_width'] = '20 um'
design.variables['cpw_gap'] = '12.25 um'

fillet='99.99um'

cpw_options = Dict(
    lead=Dict(
        start_straight='100um',
        end_straight='100um'),
    fillet=fillet
    )

main_branch_height_um= 50
main_branch_half_width_um = 800

In [5]:
gui = MetalGUI(design)
gui.rebuild()

In [7]:
# --- Resonator Components ---

otg1 = OpenToGround(design, 'otg1', options=dict(
    chip='main', pos_x='-0.5mm', pos_y='-10um', orientation=180,
    width='20um', gap='12.25um', termination_gap='12.25um'))
    
sg1 = ShortToGround(design, 'sg1', options=dict(
    chip='main', pos_x='0mm', pos_y='-0.91mm', orientation=-90,
    width='20um', gap='12.25um'))

common_kwargs = dict(
    trace_width='20um',
    trace_gap='12.25um',
    total_length=f'{total_length_mm}mm',
    hfss_wire_bonds=True,
    fillet='99.9 um',
    lead=dict(start_straight='100um', end_straight = "20um"),
    pin_inputs=Dict(
        start_pin=Dict(component='sg1', pin='short'),
        end_pin=Dict(component='otg1', pin='open')),
)
try:
    res1.delete()
except NameError : pass

if ground_pos_mm is None:
    res1 = RouteMeander(design, 'resonator1', Dict(**common_kwargs))
else:
    if not (EDGE_MARGIN_MM <= ground_pos_mm <= total_length_mm - EDGE_MARGIN_MM):
        raise ValueError(
            f"ground_pos_mm={ground_pos_mm} out of valid range "
            f"[{EDGE_MARGIN_MM}, {total_length_mm - EDGE_MARGIN_MM}]"
        )
    res1 = RouteMeanderGrounded(design, 'resonator1', Dict(
        **common_kwargs,
        ground_straps=dict(
            positions=[f'{ground_pos_mm}mm'],
            width=f'{GROUND_WIDTH_UM}um',
            overlap=f'{ground_overlap_um}um'),
    ))

gui.rebuild()

In [8]:
import geopandas as gpd
import pandas as pd

all_bounds = []
for table_name, table in design.qgeometry.tables.items():
    if len(table) == 0:
        continue
    b = table.total_bounds  # [minx, miny, maxx, maxy], in mm
    print(f"{table_name:>8}: X=[{b[0]:.4f},{b[2]:.4f}]  Y=[{b[1]:.4f},{b[3]:.4f}]")
    all_bounds.append(b)

import numpy as np
all_bounds = np.array(all_bounds)
print(f"\nOverall: X=[{all_bounds[:,0].min():.4f},{all_bounds[:,2].max():.4f}]  "
      f"Y=[{all_bounds[:,1].min():.4f},{all_bounds[:,3].max():.4f}]")

    path: X=[-0.6144,0.6144]  Y=[-0.9100,-0.0100]
    poly: X=[-0.5122,-0.1279]  Y=[-0.2362,0.0123]

Overall: X=[-0.6144,0.6144]  Y=[-0.9100,0.0123]
